# Colab Bridge: Tải Dữ Liệu Từ Google Drive Lên Kaggle Dataset
### Tận dụng băng thông Google Cloud tốc độ cao (200MB/s - 1GB/s)
Notebook này cho phép bạn chuyển toàn bộ các tệp video `.zip` hoặc `.blob` từ **Google Drive** trực tiếp lên **Kaggle Datasets** một cách tự động, nhanh chóng và không chiếm dung lượng ổ cứng máy tính cá nhân.

In [ ]:
# 1. MOUNT GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')
print("Đã kết nối Google Drive thành công!")

In [ ]:
# 2. CÀI ĐẶT VÀ CẤU HÌNH KAGGLE API TOKEN
import os
import json
import shutil
from pathlib import Path

# Đặt file kaggle.json trong Google Drive hoặc upload trực tiếp lên Colab
KAGGLE_JSON_PATH = Path("/content/drive/MyDrive/kaggle.json") 

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
dest_json = kaggle_dir / "kaggle.json"

if KAGGLE_JSON_PATH.exists():
    shutil.copy(str(KAGGLE_JSON_PATH), str(dest_json))
    os.chmod(str(dest_json), 0o600)
    print("Đã cấu hình Kaggle API Token từ Google Drive!")
else:
    print("Chưa tìm thấy kaggle.json trên Drive. Vui lòng tải file kaggle.json lên /content/ rồi chạy lệnh dưới:")
    # shutil.copy("/content/kaggle.json", str(dest_json))
    # os.chmod(str(dest_json), 0o600)

!pip install -q kaggle

In [ ]:
# 3. CHỌN DỮ LIỆU TỪ GOOGLE DRIVE VÀ CHUẨN BỊ STAGING
# Đường dẫn thư mục chứa video trên Google Drive
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/AIC2026/raw_videos")
STAGING_DIR = Path("/content/kaggle_staging")
STAGING_DIR.mkdir(parents=True, exist_ok=True)

# Tên định danh Dataset trên Kaggle
DATASET_SLUG = "aic2026-batch1-videos"
DATASET_TITLE = "AIC 2026 Batch 1 Raw Videos"

# Copy hoặc tạo symlink các file zip cần tải lên
print(f"Đang quét dữ liệu từ {DRIVE_DATA_DIR}...")
zip_files = list(DRIVE_DATA_DIR.glob("*.zip")) + list(DRIVE_DATA_DIR.glob("*.blob"))
print(f"Tìm thấy {len(zip_files)} file:")
for f in zip_files:
    print(f"  - {f.name} ({f.stat().st_size / (1024*1024):.2f} MB)")
    # Copy vào staging để upload
    shutil.copy(str(f), str(STAGING_DIR / f.name))

# Tạo file dataset-metadata.json
import kaggle
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi()
api.authenticate()

username = api.get_config_value('username')
metadata = {
    "title": DATASET_TITLE,
    "id": f"{username}/{DATASET_SLUG}",
    "licenses": [{"name": "CC0-1.0"}]
}
with open(STAGING_DIR / "dataset-metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("Đã hoàn thành chuẩn bị staging!")

In [ ]:
# 4. ĐẨY DỮ LIỆU LÊN KAGGLE DATASET
print(f"Bắt đầu đẩy dữ liệu lên Kaggle Dataset: {metadata['id']}...")

# Kiểm tra xem dataset đã tồn tại hay chưa
try:
    api.dataset_create_new(
        folder=str(STAGING_DIR),
        public=False,
        quiet=False
    )
    print(f"TẠO MỚI THÀNH CÔNG: https://www.kaggle.com/datasets/{metadata['id']}")
except Exception as e:
    print(f"Dataset đã tồn tại hoặc lỗi: {e}")
    print("Tiến hành cập nhật phiên bản mới...")
    api.dataset_create_version(
        folder=str(STAGING_DIR),
        version_notes="Cập nhật dữ liệu tự động từ Colab",
        quiet=False
    )
    print(f"CẬP NHẬT VERSION THÀNH CÔNG: https://www.kaggle.com/datasets/{metadata['id']}")